# Etapa 01 — aquisição Bronze

Autor: Wanderson Ferreira. Marco: 2026-10-06.
Branch da implementação: `feat/bronze-extraction`.
Revisão avaliada: `18242fcc8e340bad52a394c7c3624b78bd809b6c`.

Este notebook registra decisões e evidências da etapa. A síntese está no
[notebook principal](../fraud_detection_mlops.ipynb); o contrato está em
[Data pipeline](../../docs/DATA_PIPELINE.md), e os comandos completos em
[Operations](../../docs/OPERATIONS.md).

As células opcionais abaixo leem apenas o inventário e o recibo versionados.
Não foram executadas para preencher este documento e não contêm outputs.
Podem ser executadas com um kernel Python; nenhuma aquisição de dados é iniciada.


## Contexto e decisão de fonte

Precisávamos de dados temporais para um projeto de classificação de fraude,
permitindo preservar a ordem dos eventos nas futuras etapas de avaliação.
Escolhemos o dataset simulado do Fraud Detection Handbook e fixamos o commit
`6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a`.

São 183 arquivos diários `.pkl` de 2018-04-01 a 2018-09-30. Sua identidade está
no [inventário](../../references/handbook_source.json). A simulação oferece uma
base educacional; conclusões sobre fraude real exigirão outras evidências.
Os termos conhecidos da fonte estão registrados no inventário.


## Decisões da implementação

| Decisão | Motivo e consequência |
| --- | --- |
| Commit completo na URL | Repetir a fonte, sem depender de alterações futuras de branch |
| Bytes originais preservados | Manter a Bronze como referência para transformações |
| Tamanho e Git blob SHA-1 | Conferir a identidade contra o inventário da fonte |
| SHA-256 local | Conferir integridade na reutilização e verificação posterior |
| Temporários e atualização por arquivo | Retomar uma aquisição sem aceitar downloads incompletos |
| Manifesto e auditoria por execução | Explicar origem, cobertura e resultado operacional |
| Filesystem local | Começar com uma base acessível e reproduzível |

O extrator é mais extenso que um downloader simples porque implementa os controles
ao redor da aquisição. Ele será usado sobretudo para reprodução e recuperação;
treinamentos futuros trabalharão sobre dados preparados. O verificador offline
poderá conferir a base antes das transformações.


## Como ler o algoritmo

A sequência principal é validar inventário e intervalo, obter o lock, conferir o
manifesto e arquivos anteriores, adquirir ou reutilizar cada partição e registrar
os resultados. Uma falha capturada deixa as partições já aceitas disponíveis para
retomada e uma auditoria com o erro, quando a gravação é possível.

Para estudar o módulo, comece por `extract_bronze` e depois percorra os grupos de
funções descritos em [Data pipeline](../../docs/DATA_PIPELINE.md#leitura-do-algoritmo-por-responsabilidade).
O download não desserializa os pickles. A inspeção de transações será uma etapa
distinta, anterior à definição do contrato semântico da Silver.


In [1]:
import json
from pathlib import Path

candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (path for path in candidates if (path / "references/handbook_source.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Execute a partir do checkout deste projeto.")

inventory_path = project_root / "references/handbook_source.json"
inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
print("Fonte:", inventory["source"]["repository"])
print("Commit:", inventory["source"]["commit"])
print("Partições esperadas:", len(inventory["files"]))
print("Período:", inventory["files"][0]["date"], "a", inventory["files"][-1]["date"])
print("Bytes esperados:", sum(item["size_bytes"] for item in inventory["files"]))

Fonte: Fraud-Detection-Handbook/simulated-data-raw
Commit: 6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a
Partições esperadas: 183
Período: 2018-04-01 a 2018-09-30
Bytes esperados: 107121710


## Evidências locais informadas pelo autor

Estas saídas foram fornecidas pelo autor no terminal. Não são outputs de execução
das células deste notebook, e os JSONs locais de auditoria não foram abertos para
preparar o recibo documental.

| Execução | Resultado observado |
| --- | --- |
| Amostra de sete dias, run `96b07857a7164f39839482c048640b52` | `Downloaded: 7; skipped: 0` |
| Verificação da amostra | `Verified: 7/183; complete: False` |
| Aquisição completa, run `e03daaf677b4486e9081d2d1e652638c` | `Downloaded: 176; skipped: 7` |
| Verificação completa | `Verified: 183/183; complete: True` |
| Checks locais | Lockfile, lint e formatação aprovados; `39 passed in 0.29s` |

Ambiente de referência anteriormente informado pelo autor: Python 3.14.4 e
Poetry 2.4.3. A execução completa demonstra reutilização dos sete arquivos
iniciais. Falhas e recuperação também são cobertas pelos testes automatizados;
não há aqui um ensaio local relatado de cada cenário operacional.


## Evidência da CI

[Execução 37514553686](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37514553686):
workflow `CI`, evento `push`, commit `18242fcc8e340bad52a394c7c3624b78bd809b6c`,
estado `completed` e conclusão `success`, conferidos via consulta de leitura.

A CI confere ambiente, instalação, lint, formatação e testes. Os downloads dos
testes são simulados; a CI não comprova a aquisição real das 183 partições.


In [2]:
import hashlib

receipt_path = project_root / "references/evidence/bronze_2026-10-06.json"
receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
inventory_sha256 = hashlib.sha256(inventory_path.read_bytes()).hexdigest()
if inventory_sha256 != receipt["source"]["inventory_sha256"]:
    raise ValueError("O inventário atual difere do recibo histórico da Bronze.")
print("Implementação avaliada:", receipt["implementation"]["commit"])
print("Verificação local informada:", receipt["local_evidence"]["full_verification"]["stdout"])
print("CI registrada:", receipt["ci_evidence"]["url"])
print("O recibo histórico não verifica o snapshot local desta máquina.")

Implementação avaliada: 18242fcc8e340bad52a394c7c3624b78bd809b6c
Verificação local informada: Verified: 183/183; complete: True
CI registrada: https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37514553686
O recibo histórico não verifica o snapshot local desta máquina.


## Limites e passagem para a Silver

O recibo relaciona evidências; não substitui arquivos brutos, manifesto ou auditorias
nativas. O armazenamento é local, sem assinatura de metadados ou auditoria imutável.
A identidade da fonte e a integridade dos arquivos estão verificadas no resultado
informado. Schema, nulos, duplicatas, rótulos e consistência das transações ainda
precisam de inspeção.

A próxima etapa definirá esse contrato antes de converter para Parquet e consultar
com DuckDB. Os resultados detalhados entrarão em notebook próprio, com síntese no
principal. A [política](../../docs/DOCUMENTATION_POLICY.md) acompanha esse fluxo.
